In [1]:
pip install transformers[torch] datasets pythainlp pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 83.5 MB/s eta 0:00:00


In [7]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- 1. Load & Sample Data (ปรับเพิ่มเป็น 3,000 แถว) ---
df = pd.read_csv('phishing_email.csv')
df = df.dropna().drop_duplicates()

# เพิ่มจำนวนเป็น 3000 เพื่อให้ Accuracy สูงขึ้นแต่ยังเทรนไว
df = df.sample(n=3000, random_state=42).reset_index(drop=True)

# --- 2. Setup Model & Tokenizer ---
model_name = "distilbert-base-uncased" # ตัวเล็กแต่ฉลาด
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text_combined"], padding="max_length", truncation=True, max_length=512)

# --- 3. Prepare Dataset ---
dataset = Dataset.from_pandas(df[['text_combined', 'label']])
dataset = dataset.rename_column("label", "labels")

train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
tokenized_datasets = train_test_split.map(tokenize_function, batched=True)

# --- 4. Metrics Function ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

# --- 5. Model & Training Config ---
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./optimized_bert",
    learning_rate=2e-5,              # ปรับ Learning Rate ให้เล็กลงเพื่อให้เรียนรู้ละเอียดขึ้น
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,              # เพิ่มเป็น 2 รอบ
    weight_decay=0.01,
    eval_strategy="epoch",           # ให้โชว์ผลลัพธ์ทุกรอบที่จบ
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# --- 6. Train & Evaluate ---
print("🚀 กำลังเริ่มเทรนแบบ Optimized (คาดว่าใช้เวลา 5-7 นาที)...")
trainer.train()

print("\n📊 ผลลัพธ์หลังปรับปรุง:")
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 กำลังเริ่มเทรนแบบ Optimized (คาดว่าใช้เวลา 5-7 นาที)...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.120560,0.958333,0.957118
2,No log,0.112634,0.965000,0.963855



📊 ผลลัพธ์หลังปรับปรุง:


Accuracy: 0.9650
F1 Score: 0.9639
